# EFIplus Mediterranean Dataset — Linear Regression Exercise

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FernandaChacara/PML/blob/main/code-scripts/EFIplus_medit_regression_notebook.ipynb)

This notebook answers the assignment using the dataset `EFIplus_medit.zip`.

The analysis has three parts:

1. Simple linear regressions between species richness and each continuous environmental variable.
2. Multiple linear regression using all predictors and comparison with the univariate coefficients.
3. Multicollinearity diagnosis and a second, more parsimonious regression model.

## Environmental predictors used

The predictors required by the assignment are:

- `Altitude`
- `Actual_river_slope`
- `Elevation_mean_catch`
- `prec_ann_catch`
- `temp_ann`
- `temp_jan`
- `temp_jul`

The response variable is **species richness**.

In [ ]:

import os
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.linear_model import LinearRegression
from sklearn.inspection import PartialDependenceDisplay

from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

# 1. Load the dataset

This notebook expects `EFIplus_medit.zip` to be in the working directory.

In Colab, upload the ZIP file using the file panel on the left. In VS Code, place the ZIP file in the same folder as this notebook.

In [ ]:

zip_path = "EFIplus_medit.zip"
extract_folder = "EFIplus_medit"

if not os.path.exists(zip_path):
    raise FileNotFoundError(
        "EFIplus_medit.zip was not found. Upload it to the notebook working directory before running this cell."
    )

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

all_files = []
for root, dirs, files in os.walk(extract_folder):
    for file in files:
        all_files.append(os.path.join(root, file))

print("Files extracted:")
for file in all_files:
    print(file)

data_files = [f for f in all_files if f.lower().endswith((".csv", ".txt"))]

if len(data_files) == 0:
    raise FileNotFoundError("No CSV or TXT file was found inside the ZIP file.")

csv_file = data_files[0]
print("\nSelected data file:", csv_file)

In [ ]:
# prompt: Read the selected EFIplus Mediterranean dataset file robustly.

try:
    df = pd.read_csv(csv_file)
    if df.shape[1] == 1:
        df = pd.read_csv(csv_file, sep=";")
except Exception:
    df = pd.read_csv(csv_file, sep=";")

print("Dataset shape:", df.shape)
df.head()

# 2. Define response variable and predictors

The code tries to automatically detect the species richness column. If it fails, inspect `df.columns` and manually set the variable `response`.

In [ ]:

predictors_original = [
    "Altitude",
    "Actual_river_slope",
    "Elevation_mean_catch",
    "prec_ann_catch",
    "temp_ann",
    "temp_jan",
    "temp_jul"
]

missing_predictors = [col for col in predictors_original if col not in df.columns]
if missing_predictors:
    raise ValueError(
        f"The following required predictor columns are missing: {missing_predictors}\n"
        f"Available columns are: {list(df.columns)}"
    )

possible_response_names = [
    "Species_richness", "species_richness", "Richness", "richness",
    "Fish_richness", "fish_richness", "sp_richness", "n_species",
    "SpeciesRichness", "speciesRichness"
]

response = None

for col in possible_response_names:
    if col in df.columns:
        response = col
        break

if response is None:
    richness_candidates = [col for col in df.columns if "rich" in col.lower()]
    print("Possible richness columns:", richness_candidates)
    if len(richness_candidates) == 1:
        response = richness_candidates[0]
    else:
        raise ValueError(
            "Could not automatically identify the species richness column. "
            "Please inspect df.columns and manually set response = 'your_column_name'."
        )

print("Response variable:", response)
print("Predictor variables:", predictors_original)

In [ ]:

model_data = df[[response] + predictors_original].copy()

for col in model_data.columns:
    model_data[col] = pd.to_numeric(model_data[col], errors="coerce")

print("Missing values before dropping rows:")
display(model_data.isna().sum())

model_data = model_data.dropna()

print("\nShape after dropping missing values:", model_data.shape)
display(model_data.describe())

# 3. Check whether transformations are needed

Before fitting the regressions, I check the distributions of the continuous predictors.

A transformation is useful when a predictor is strongly skewed. In this notebook, I use this practical rule:

- if a non-temperature predictor is non-negative and has absolute skewness greater than 1, I apply `log1p`;
- temperature variables are kept in their original units for easier interpretation.

In [ ]:
# prompt: Check skewness and plot predictor distributions.

skew_table = []

for var in predictors_original:
    skew_table.append({
        "Variable": var,
        "Min": model_data[var].min(),
        "Max": model_data[var].max(),
        "Skewness": model_data[var].skew()
    })

skew_df = pd.DataFrame(skew_table)
display(skew_df)

for var in predictors_original:
    plt.figure(figsize=(6, 4))
    plt.hist(model_data[var], bins=30)
    plt.title(f"Distribution of {var}")
    plt.xlabel(var)
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
# prompt: Apply log1p transformation to strongly skewed positive predictors.

df_reg = model_data.copy()

transformed_predictors = []
transformation_report = []

for var in predictors_original:
    skewness = df_reg[var].skew()
    min_value = df_reg[var].min()
    is_temperature = var in ["temp_ann", "temp_jan", "temp_jul"]

    if (not is_temperature) and (min_value >= 0) and (abs(skewness) > 1):
        new_var = f"log1p_{var}"
        df_reg[new_var] = np.log1p(df_reg[var])
        transformed_predictors.append(new_var)
        transformation_report.append({
            "Original variable": var,
            "Used variable": new_var,
            "Transformation": "log1p",
            "Original skewness": skewness,
            "Transformed skewness": df_reg[new_var].skew()
        })
    else:
        transformed_predictors.append(var)
        transformation_report.append({
            "Original variable": var,
            "Used variable": var,
            "Transformation": "none",
            "Original skewness": skewness,
            "Transformed skewness": skewness
        })

transformation_df = pd.DataFrame(transformation_report)

print("Variables used in the regressions:")
print(transformed_predictors)
display(transformation_df)

# 4. Simple linear regressions

Each simple regression relates species richness to one environmental predictor at a time.

For each model I compute:

- coefficient estimate;
- intercept;
- R-squared;
- F-statistic;
- F-test p-value.

In [ ]:

simple_results = []

for var in transformed_predictors:
    formula = f"{response} ~ Q('{var}')"
    model = smf.ols(formula, data=df_reg).fit()

    coef_name = f"Q('{var}')"
    simple_results.append({
        "Predictor": var,
        "Intercept": model.params["Intercept"],
        "Coefficient_simple": model.params[coef_name],
        "R_squared": model.rsquared,
        "F_statistic": model.fvalue,
        "F_pvalue": model.f_pvalue,
        "Coefficient_pvalue": model.pvalues[coef_name]
    })

simple_results_df = pd.DataFrame(simple_results)
display(simple_results_df)

In [ ]:

for var in transformed_predictors:
    model = smf.ols(f"{response} ~ Q('{var}')", data=df_reg).fit()

    x = df_reg[var]
    y = df_reg[response]

    x_grid = np.linspace(x.min(), x.max(), 100)
    y_pred = model.params["Intercept"] + model.params[f"Q('{var}')"] * x_grid

    plt.figure(figsize=(6, 4))
    plt.scatter(x, y, alpha=0.5)
    plt.plot(x_grid, y_pred)
    plt.xlabel(var)
    plt.ylabel(response)
    plt.title(f"Simple regression: {response} ~ {var}")
    plt.show()

# 5. Multiple linear regression

Now I fit a multiple linear regression with all predictors simultaneously.

In the simple regressions, each coefficient describes the isolated relationship between richness and one predictor.

In the multiple regression, each coefficient represents the effect of one predictor while controlling for all the others. Therefore, coefficients may change in magnitude, significance or even sign.

In [ ]:
# prompt: Fit a multiple linear regression using all transformed environmental predictors.

formula_multiple = response + " ~ " + " + ".join([f"Q('{v}')" for v in transformed_predictors])

multiple_model = smf.ols(formula_multiple, data=df_reg).fit()

print(multiple_model.summary())

In [ ]:


multiple_coefs = []

for var in transformed_predictors:
    coef_name = f"Q('{var}')"
    multiple_coefs.append({
        "Predictor": var,
        "Coefficient_multiple": multiple_model.params[coef_name],
        "Multiple_pvalue": multiple_model.pvalues[coef_name]
    })

multiple_coefs_df = pd.DataFrame(multiple_coefs)

coefficient_comparison = simple_results_df.merge(multiple_coefs_df, on="Predictor", how="left")
coefficient_comparison["Coefficient_change"] = (
    coefficient_comparison["Coefficient_multiple"] - coefficient_comparison["Coefficient_simple"]
)

display(coefficient_comparison[[
    "Predictor",
    "Coefficient_simple",
    "Coefficient_multiple",
    "Coefficient_change",
    "R_squared",
    "F_statistic",
    "F_pvalue",
    "Coefficient_pvalue",
    "Multiple_pvalue"
]])

# 6. Partial dependence plots

Partial dependence plots show the partial response of species richness to each predictor while the other predictors are accounted for.

For a linear model, the partial dependence response is approximately linear, but the slope corresponds to the coefficient estimated in the multiple regression.

In [ ]:
# prompt: Create partial dependence plots for the multiple linear regression model.

X = df_reg[transformed_predictors]
y = df_reg[response]

sklearn_lm = LinearRegression()
sklearn_lm.fit(X, y)

for var in transformed_predictors:
    fig, ax = plt.subplots(figsize=(6, 4))
    PartialDependenceDisplay.from_estimator(
        sklearn_lm,
        X,
        features=[var],
        ax=ax
    )
    plt.title(f"Partial dependence plot: {var}")
    plt.show()

# 7. Check multicollinearity

Environmental predictors can be correlated with each other.

For example, altitude, catchment elevation and temperature may represent related environmental gradients.

I check multicollinearity using:

1. a correlation matrix;
2. Variance Inflation Factor, or VIF.

A common rule of thumb is:

- VIF below 5: acceptable;
- VIF above 5: possible multicollinearity;
- VIF above 10: strong multicollinearity.

In [ ]:
# prompt: Compute and plot the correlation matrix among predictors.

corr_matrix = df_reg[transformed_predictors].corr()

display(corr_matrix)

plt.figure(figsize=(8, 6))
plt.imshow(corr_matrix, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(transformed_predictors)), transformed_predictors, rotation=90)
plt.yticks(range(len(transformed_predictors)), transformed_predictors)
plt.title("Correlation matrix among predictors")
plt.tight_layout()
plt.show()

In [ ]:

def compute_vif(data, predictors):
    X_vif = sm.add_constant(data[predictors])
    rows = []
    for i, col in enumerate(X_vif.columns):
        if col != "const":
            rows.append({
                "Predictor": col,
                "VIF": variance_inflation_factor(X_vif.values, i)
            })
    return pd.DataFrame(rows).sort_values("VIF", ascending=False)

vif_df = compute_vif(df_reg, transformed_predictors)
display(vif_df)

# 8. Parsimonious regression model

A parsimonious model uses fewer predictors and avoids including variables that are strongly redundant.

Here I use an automatic VIF-based procedure:

1. Start with all predictors.
2. Compute VIF.
3. Remove the predictor with the highest VIF if it is above 5.
4. Repeat until all remaining VIF values are below or equal to 5.

This is a transparent rule for reducing multicollinearity.

In [ ]:
# prompt: Select a parsimonious predictor set by iteratively removing variables with high VIF.

def select_by_vif(data, predictors, threshold=5.0):
    selected = predictors.copy()
    history = []

    while True:
        vif_current = compute_vif(data, selected)
        max_vif = vif_current["VIF"].max()
        worst_predictor = vif_current.iloc[0]["Predictor"]

        history.append({
            "Step": len(history) + 1,
            "Removed predictor": None if max_vif <= threshold else worst_predictor,
            "Max VIF": max_vif,
            "Predictors kept": selected.copy()
        })

        if max_vif <= threshold or len(selected) <= 2:
            break

        selected.remove(worst_predictor)

    return selected, pd.DataFrame(history)

parsimonious_predictors, vif_selection_history = select_by_vif(
    df_reg,
    transformed_predictors,
    threshold=5.0
)

print("Predictors kept in the parsimonious model:")
print(parsimonious_predictors)

print("\nVIF selection history:")
display(vif_selection_history)

print("\nFinal VIF values:")
display(compute_vif(df_reg, parsimonious_predictors))

In [ ]:

formula_parsimonious = response + " ~ " + " + ".join([f"Q('{v}')" for v in parsimonious_predictors])

parsimonious_model = smf.ols(formula_parsimonious, data=df_reg).fit()

print(parsimonious_model.summary())

In [ ]:

model_comparison = pd.DataFrame({
    "Model": ["Full multiple model", "Parsimonious model"],
    "Number of predictors": [len(transformed_predictors), len(parsimonious_predictors)],
    "R_squared": [multiple_model.rsquared, parsimonious_model.rsquared],
    "Adjusted_R_squared": [multiple_model.rsquared_adj, parsimonious_model.rsquared_adj],
    "F_statistic": [multiple_model.fvalue, parsimonious_model.fvalue],
    "F_pvalue": [multiple_model.f_pvalue, parsimonious_model.f_pvalue],
    "AIC": [multiple_model.aic, parsimonious_model.aic],
    "BIC": [multiple_model.bic, parsimonious_model.bic]
})

display(model_comparison)

In [ ]:

full_coefs = multiple_model.params.rename("Full_model_coefficient")
parsimonious_coefs = parsimonious_model.params.rename("Parsimonious_model_coefficient")

coef_model_comparison = pd.concat([full_coefs, parsimonious_coefs], axis=1)
display(coef_model_comparison)

# 9. Final interpretation template

The simple regressions show the isolated relationship between species richness and each environmental predictor.

The multiple regression coefficients can differ from the simple regression coefficients because the multiple model estimates the effect of each predictor while controlling for the remaining predictors.

If some predictors are strongly correlated, the multiple regression coefficients may become unstable. This is why the correlation matrix and VIF values are important.

After removing highly collinear predictors, the parsimonious model should be easier to interpret. The comparison between the full and parsimonious models should consider:

- R-squared;
- adjusted R-squared;
- F-statistic;
- AIC;
- BIC;
- changes in the coefficients.

A good parsimonious model should keep a reasonable explanatory power while reducing redundancy among predictors.